# Building a first Classification NN

Use the Ames Mutagenicity dataset (from assignment 1A) and build a binary classifier NN. Play with the model parameters. 

For comparison of the NN model performance, consider the performance of other (baseline) classifier models (assignment 1A):
- KNN: Test-Accuracy 0.79, Test-ROC-AUC 0.86
- Decision Tree: Test-Accuracy 0.78, Test-ROC-AUC 0.77
- Random Forest: Test-Accuracy 0.83, Test-ROC-AUC 0.90
- Gradient Boosting: Test-Accuracy 0.77, Test-ROC-AUC 0.85


#### Tasks:
1) Load the dataset `ames_data.csv`. The dataset does not contain any duplicates or NaNs
2) Feature engineering: Calculate various fingerprints from the SMILES strings via mol objects using RDKit(snippet provided for Morgan FPs and MACCS keys)
3) Create feature matrix and target vector. Choose first the MorganFP (Later repeat the process for other fingerprint types). Convert the training and test sets into pytorch tensors.
4) Build your NN (see below for more info)
5) Train your model on the Morgan Fingerprints (and repeat later for other FP types)
6) Evaluate your model's performance and compare to other classifier models
7) Save the model / current state.
8) Respond to the discussion points


#### Note:
The aim of this exercise is to gain a bit of practice in building a simple NN and to see how different parameters and feature engineering influence the model. Maximum accuracy is not the target. 

There is no need to venture too far into the details or more advanced approaches just yet (e.g. batched training would be complete overkill for this assignment - we will discuss that in the next sessions)

0) Import dependencies and datasets

In [7]:
# complete imports if needed for your solution
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem import MACCSkeys

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim

1) Load and investigate the data

In [8]:
df = pd.read_csv("ames_data.csv")
df.head()

,drug_id,smiles,mutagenicity
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1


2) Generate different fingerprints (try at least one additional FP type as provided in RDKit and use two different fpSizes on one of them) - all of them will be saved in new columns in the Dataframe.

In [9]:
from rdkit.Chem import RDKFingerprint


def smiles_to_mol(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return mol

def morganfp(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol)
    return np.array(fp)

def morganfp_1024(mol):
    fp = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024).GetFingerprint(mol)
    return np.array(fp)

def maccskeys(mol):
    fp = MACCSkeys.GenMACCSKeys(mol)
    return np.array(fp)


def rdkfingerprint(mol):
    fp = RDKFingerprint(mol, fpSize=2048)
    return np.array(fp)

fpgens = {
    "MorganFP": morganfp,
    "MorganFP_1024": morganfp_1024,
    "MACCSkeys": maccskeys,
    "RDKFingerprint": rdkfingerprint
}

df["mol"] = df["smiles"].apply(smiles_to_mol)

for name, fpgen in fpgens.items():
    df[name] = df["mol"].apply(fpgen)
df.head()

,drug_id,smiles,mutagenicity,mol,MorganFP,MorganFP_1024,MACCSkeys,RDKFingerprint
0,Drug 0,O=[N+]([O-])c1ccc2ccc3ccc([N+](=O)[O-])c4c5ccc...,1,<rdkit.Chem.rdchem.Mol object at 0x0000019C14B...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."
1,Drug 1,O=[N+]([O-])c1c2c(c3ccc4cccc5ccc1c3c45)CCCC2,1,<rdkit.Chem.rdchem.Mol object at 0x0000019C14B...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, ..."
2,Drug 2,O=c1c2ccccc2c(=O)c2c1ccc1c2[nH]c2c3c(=O)c4cccc...,0,<rdkit.Chem.rdchem.Mol object at 0x0000019C14B...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, ..."
3,Drug 3,[N-]=[N+]=CC(=O)NCC(=O)NN,1,<rdkit.Chem.rdchem.Mol object at 0x0000019C14B...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,Drug 4,[N-]=[N+]=C1C=NC(=O)NC1=O,1,<rdkit.Chem.rdchem.Mol object at 0x0000019C14B...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


3. Create feature matrix and target vector. The snippet below converts the data into numpy arrays. Start with the Morgan Fingerprints (and later return here to apply your modell to different fingerprint types - not all of the fingerprints may have the same length, so you may have to adapt the width of your layers).

Do a train startified test split and convert into pytorch tensors.

In [276]:
X = np.stack(df["RDKFingerprint"].values) # joins multiple arrays along a new axis - builds a proper array from the line-by-line arrays in the df.
y = df["mutagenicity"].to_numpy()

In [277]:
# Train Test Split and coverting to tensors

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

4) Build the NN - adhere to some robust standard values. Start simple and train the model on Morgan FP first.

Optimise the model parameters based on observed over-/underfitting. Experiment with different width and depth, as well as other model parameters. Explore some options to prevent overfitting, e.g. Early stopping (e.g. manually by limiting the epochs) or dropouts. 

Note: Since the input layer needs a lot of neurons (e.g. 2048 bit in the MFPs), consider shrinking the widht from layer to layer. 

Hint: If you use `BCELoss()` as loss function, combine it with a `sigmoid` activation in the last layer. If you use `BCEWithLogitsLoss()`, do not specify any activation in the forward pass (`x = self.outputlayer(x)`).

In [278]:
class BinClassifierNN(nn.Module):
    def __init__(self):
        super(BinClassifierNN, self).__init__()
        # define your model width and depth below
        self.fc1 = nn.Linear(input_size, hidden_size) 
        self.fc2 = nn.Linear(hidden_size, hidden_size)   
        self.fc3 = nn.Linear(hidden_size, hidden_size)  
        self.fc4 = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(0.3) # probability of dropping a nuron during training, prevents overfitting and relying too much on specific neurons

    def forward(self, x):
        # Specify the forward pass, i.e. activation functions.
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = self.dropout(x)
        x = torch.sigmoid(self.fc4(x)) # applying sigmoid in combination with BCELoss as stated above
        return x


In [279]:
# Parameters (change and add as needed)
input_size = X_train.shape[1]
hidden_size = 40
output_size = 1
learning_rate = 0.002
num_epochs = 100

In [280]:
model = BinClassifierNN()

# choose a loss function for the classification
criterion = nn.BCELoss() 

# choose an optimizer
optimizer = optim.Adam(model.parameters(), learning_rate)

5. Train the NN. Note that you may have to squeeze the output (`outputs=models(X_train).squeeze`). This will reduce the actual output of the shape ``[N, 1]`` to ``[N]``, which is comparable to y (The final layer naturally produces a column tensor, which is not directly comparable to the 1D target tensor).

In [281]:
for epoch in range(num_epochs):
    
    model.train()
    outputs = model(X_train).squeeze()
    loss = criterion(outputs, y_train)

    optimizer.zero_grad() # clear existing gradients
    loss.backward() # backpropagation of loss function to optimise gradient
    optimizer.step() # update parameters using Adam

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch + 1}/100], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.6198
Epoch [20/100], Loss: 0.5082
Epoch [30/100], Loss: 0.4179
Epoch [40/100], Loss: 0.3703
Epoch [50/100], Loss: 0.3043
Epoch [60/100], Loss: 0.2478
Epoch [70/100], Loss: 0.2035
Epoch [80/100], Loss: 0.1576
Epoch [90/100], Loss: 0.1142
Epoch [100/100], Loss: 0.3037


6) Evaluate the model. As a first metric, you can use the same loss function to evaluate the model on the test set. For better comparison with methods tested in the assignment 1A (results vide supra), use metrics from scikit-learn (e.g. the accuracy or ROC-AUC score).

Hint: for your prediction you may have to use `squeeze` again to match the target vector in the test set (e.g. ``y_pred = model(X_test).squeeze()``)

In [282]:
# Evaluate the model
from sklearn.metrics import accuracy_score, roc_auc_score

mse_loss = nn.BCELoss()

model.eval()

with torch.no_grad():
    train = model(X_train).squeeze()
    test = model(X_test).squeeze()

    train_pred = (train > 0.5).float() # cutoff at 0.5 to get binary predictions
    test_pred = (test > 0.5).float()

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Train ROC-AUC:", roc_auc_score(y_train, train))
print("Test Accuracy:", accuracy_score(y_test, test_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, test))

Train Accuracy: 0.8337341119890073
Train ROC-AUC: 0.953793128642445
Test Accuracy: 0.7657967032967034
Test ROC-AUC: 0.8491917483678803


7) Research how you can save the model / current state for later reuse. What are different options here? How can it be loaded again?

In [ ]:
# https://docs.pytorch.org/tutorials/beginner/saving_loading_models

# 1. torch.save: Saves a serialized object to disk. This function uses Python’s pickle utility for serialization. Models, tensors, and dictionaries of all kinds of objects can be saved using this function.

torch.save(model.state_dict(), "bin_classifier_RDKFingerprint.pth") # saving

Reload using:

model = BinClassifierNN()

model.load_state_dict(torch.load("XXX.pth", weights_only=True))

model.eval()


#### 8) Discussion points
1) How did your model compare to other simple ML classifiers (all used the Morgan FPs)? Discuss!
2) Did you observe any difference between different fingerprint types?
3) Did the fingerprint size impact the model prediction? What message is to be learned from this?
4) What were some model parameters for decent performance depending on the fingerprint type? 
5) Was overfitting a problem? What approaches did you apply to limit that issue? What else would be possible
6) Consider the target "mutagenicity" in the context of molecular structure. What does noise mean here? How could you use such a predictive model in the lab? What other data-driven tools could be interesting in this QSAR context?
7) Why is exporting a full model usually not recommended?

1. They were all slightly better than the simple ML classifiers except to RF. RF has very similar results, maybe even slightly better ones compared to some. NN worked well but wasnt clearly better than simple ML classifiers. Maybe this is due to Morgan fingerprints already having a decent scaffold of information which simple ML classifiers can work on.
2. Yes, with the biggest differene being the hidden layer and the hidden_size between Morgan and the other two. Having two hidden layers with Morgan made the overfitting A LOT worse but this wasnt the case for MACCS or RDK, it actually improved the outcome.
3. No, not really but 2024 did give slightly better results. Maybe the difference would have been a lot bigger with even less than 1024.
4. I optimized first using **MorganFP (used same parameters for 1024)**. Adding the dropout value definitely helped the model (changing values from 0.1-0.3 did not really make a big difference). Also only using one hidden layer was A LOT better than two, because two gave really bad overfitting. Increasing epochs also just gave worse overfitting and paramters used at end were kind of the sweet spot. Also increasing or decreasing hidden_size did not really improbe the model anymore. - For **RDKFingerprints** & **MACCSkeys** I ended up with very similar paramters. Both showed a lot better results having two hidden layers instead of one as MorganFP as well as an increased hidden_size to 40. Changing other paramters did not make the models better.
5. Yes at the start it was a big problem. I started using two hidden layers which gave really bad overfitting. Changing to one made it a lot better. Also addition of dropout in NN made the model better. Less epochs would also give less overfitting but I noticed the model gets worse. Increasing learning rate from 0.001 also just made overfitting that much worse so I just stayed on it.
6. I dont really know what noiose would be in this case, inconsistent labels or same molecular scaffold structure? A model like this could be used as a tool before experiments to screen different compounds which are later used in testing. Kind of like an helper in decision making of what to test. For other tools, maybe a smililarity based tool, which compares different compounds?
7. I guess if you save the full model is dependent on the exact code in the notebook? And if only used paramters are saved, they can be applied in a different notebook more easily and thus is more robust.